# Named Entity Recognition Notebook

> Hands-on Build It and Exercises.

## Build It

### Step 1: BIO tagging helpers

In [ ]:
```python

def spans_to_bio(tokens, spans):

    labels = ["O"] * len(tokens)

    for start, end, label in spans:

        labels[start] = f"B-{label}"

        for i in range(start + 1, end):

            labels[i] = f"I-{label}"

    return labels

def bio_to_spans(tokens, labels):

    spans = []

    current = None

    for i, label in enumerate(labels):

        if label.startswith("B-"):

            if current:

                spans.append(current)

            current = (i, i + 1, label[2:])

        elif label.startswith("I-") and current and current[2] == label[2:]:

            current = (current[0], i + 1, current[2])

        else:

            if current:

                spans.append(current)

                current = None

    if current:

        spans.append(current)

    return spans

In [ ]:
```

In [ ]:
```python

>>> tokens = ["Apple", "sued", "Google", "over", "iPhone", "sales", "."]

>>> labels = ["B-ORG", "O", "B-ORG", "O", "B-PRODUCT", "O", "O"]

>>> bio_to_spans(tokens, labels)

[(0, 1, 'ORG'), (2, 3, 'ORG'), (4, 5, 'PRODUCT')]

In [ ]:
```

### Step 2: hand-crafted features

For classical (non-neural) NER, features are the game. Useful ones:

In [ ]:
```python

def token_features(token, prev_token, next_token):

    return {

        "lower": token.lower(),

        "is_upper": token.isupper(),

        "is_title": token.istitle(),

        "has_digit": any(c.isdigit() for c in token),

        "suffix_3": token[-3:].lower(),

        "shape": word_shape(token),

        "prev_lower": prev_token.lower() if prev_token else "<BOS>",

        "next_lower": next_token.lower() if next_token else "<EOS>",

    }

def word_shape(word):

    out = []

    for c in word:

        if c.isupper():

            out.append("X")

        elif c.islower():

            out.append("x")

        elif c.isdigit():

            out.append("d")

        else:

            out.append(c)

    return "".join(out)

In [ ]:
```

`word_shape("iPhone")` returns `xXxxxx`. `word_shape("USA-2024")` returns `XXX-dddd`. Capitalization patterns are high-signal for proper nouns.

### Step 3: a simple rule-based + dictionary baseline

In [ ]:
```python

ORG_GAZETTEER = {"Apple", "Google", "Microsoft", "OpenAI", "Meta", "Amazon", "Netflix"}

GPE_GAZETTEER = {"US", "USA", "UK", "India", "Germany", "France"}

PRODUCT_GAZETTEER = {"iPhone", "Android", "Windows", "ChatGPT", "Claude"}

def rule_based_ner(tokens):

    labels = []

    for token in tokens:

        if token in ORG_GAZETTEER:

            labels.append("B-ORG")

        elif token in GPE_GAZETTEER:

            labels.append("B-GPE")

        elif token in PRODUCT_GAZETTEER:

            labels.append("B-PRODUCT")

        else:

            labels.append("O")

    return labels

In [ ]:
```

Production gazetteers have millions of entries scraped from Wikipedia and DBpedia. Coverage is good. Disambiguation (`Apple` the company vs the fruit) is terrible. That is why statistical models won.

### Step 4: the CRF step (sketch, not full impl)

Full CRF from scratch in 50 lines is not enlightening without the probability-theory foundations. Use `sklearn-crfsuite` instead:

In [ ]:
```python

import sklearn_crfsuite

def to_features(tokens):

    out = []

    for i, tok in enumerate(tokens):

        prev = tokens[i - 1] if i > 0 else ""

        nxt = tokens[i + 1] if i + 1 < len(tokens) else ""

        out.append({

            "word.lower()": tok.lower(),

            "word.isupper()": tok.isupper(),

            "word.istitle()": tok.istitle(),

            "word.isdigit()": tok.isdigit(),

            "word.suffix3": tok[-3:].lower(),

            "word.shape": word_shape(tok),

            "prev.word.lower()": prev.lower(),

            "next.word.lower()": nxt.lower(),

            "BOS": i == 0,

            "EOS": i == len(tokens) - 1,

        })

    return out

crf = sklearn_crfsuite.CRF(algorithm="lbfgs", c1=0.1, c2=0.1, max_iterations=100, all_possible_transitions=True)

X_train = [to_features(s) for s in sentences_tokenized]

crf.fit(X_train, bio_labels_train)

In [ ]:
```

`c1` and `c2` are L1 and L2 regularization. `all_possible_transitions=True` lets the model learn illegal sequences (e.g., `I-ORG` after `O`) are unlikely, which is how a CRF enforces BIO consistency without you writing the constraint.

### Step 5: what a BiLSTM-CRF adds

Features become learned. Inputs: token embeddings (GloVe or fastText). LSTM reads left-to-right and right-to-left. Concatenated hidden states go through a CRF output layer. The CRF still enforces tag-sequence consistency; the LSTM replaces hand-crafted features with learned ones.

In [ ]:
```python

import torch

import torch.nn as nn

class BiLSTM_CRF_Head(nn.Module):

    def __init__(self, vocab_size, embed_dim, hidden_dim, n_labels):

        super().__init__()

        self.embed = nn.Embedding(vocab_size, embed_dim)

        self.lstm = nn.LSTM(embed_dim, hidden_dim, bidirectional=True, batch_first=True)

        self.fc = nn.Linear(hidden_dim * 2, n_labels)

    def forward(self, token_ids):

        e = self.embed(token_ids)

        h, _ = self.lstm(e)

        emissions = self.fc(h)

        return emissions

In [ ]:
```

For the CRF layer, use `torchcrf.CRF` (pip install pytorch-crf). The gain over hand-crafted CRF is measurable but smaller than you expect unless you have tens of thousands of labeled sentences.

## Exercises

In [ ]:
1. **Easy.** Implement `bio_to_spans` (the inverse of `spans_to_bio`) and verify round-trip consistency on 10 sentences.
2. **Medium.** Train the sklearn-crfsuite CRF above on the CoNLL-2003 English NER dataset. Report per-entity F1 using `seqeval`. Typical result: ~84 F1.
3. **Hard.** Fine-tune `distilbert-base-cased` on a domain-specific NER dataset (medical, legal, or financial). Compare against the spaCy small model. Document data leakage checks and write up what surprised you.